# Notebook 00 — Dataset Assembly

Assemble the canonical multistation raw inputs for the Beijing Multi-Site Air Quality forecasting project.
This notebook is intentionally assembly-only and is designed for manual execution after the 12 raw PRSA station files are placed under the agreed raw path.


## 0. Imports and setup


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


In [2]:
def find_project_root(start_path: Path) -> Path:
    """Find the project root by walking upward from a start path.

    Markers searched (in priority order):
    - pyproject.toml
    - README.md
    - data/ directory

    The notebook must not assume the execution CWD is already the project root.
    """

    best_candidate = None
    best_score = -1

    for candidate in [start_path, *start_path.parents]:
        has_pyproject = (candidate / "pyproject.toml").exists()
        has_readme = (candidate / "README.md").exists()
        has_data_dir = (candidate / "data").is_dir()

        score = 0
        score += 4 if has_pyproject else 0
        score += 2 if has_readme else 0
        score += 1 if has_data_dir else 0

        if score > best_score:
            best_candidate = candidate
            best_score = score

        if has_data_dir and (has_pyproject or has_readme):
            return candidate

    if best_candidate is not None and (best_candidate / "data").is_dir():
        return best_candidate

    raise FileNotFoundError(
        "Could not locate PROJECT_ROOT by walking upward from the current working directory. "
        "Expected to find at least a 'data/' directory and ideally 'README.md' or 'pyproject.toml'."
    )


In [3]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())

SOURCE_DATASET_NAME = "UCI Beijing Multi-Site Air Quality Dataset"
SOURCE_DATASET_URL = "https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data"

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR_CANONICAL = RAW_ROOT / "beijing_multi_site_air_quality"
RAW_DATA_DIR_FALLBACK = RAW_ROOT
RAW_FILE_GLOB = "PRSA_Data_*.csv"
RAW_INPUT_PATH = "data/raw/beijing_multi_site_air_quality/PRSA_Data_*.csv"

# Canonical raw location is data/raw/beijing_multi_site_air_quality/.
# Temporary fallback: support PRSA_Data_*.csv directly under data/raw/.
raw_files_in_canonical = sorted(RAW_DATA_DIR_CANONICAL.glob(RAW_FILE_GLOB)) if RAW_DATA_DIR_CANONICAL.exists() else []
raw_files_in_fallback = sorted(RAW_DATA_DIR_FALLBACK.glob(RAW_FILE_GLOB)) if RAW_DATA_DIR_FALLBACK.exists() else []

if raw_files_in_canonical:
    RAW_DATA_DIR = RAW_DATA_DIR_CANONICAL
    RAW_DATA_DIR_MODE = "canonical"
elif raw_files_in_fallback:
    RAW_DATA_DIR = RAW_DATA_DIR_FALLBACK
    RAW_DATA_DIR_MODE = "fallback"
else:
    RAW_DATA_DIR = RAW_DATA_DIR_CANONICAL
    RAW_DATA_DIR_MODE = "canonical (not found yet)"

RAW_DATA_DIR_NOTE = (
    "Canonical raw path is data/raw/beijing_multi_site_air_quality/. "
    "Fallback mode is only for temporary compatibility when PRSA_Data_*.csv are placed directly under data/raw/."
)

INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "beijing_air_quality"
ASSEMBLED_PARQUET_PATH = INTERIM_DIR / "beijing_multisite_assembled.parquet"
ASSEMBLY_MANIFEST_PATH = INTERIM_DIR / "assembly_manifest.json"

EXPECTED_FILE_COUNT = 12

EXPECTED_SCHEMA_WITH_STATION = [
    "No", "year", "month", "day", "hour", "PM2.5", "PM10", "SO2", "NO2",
    "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM", "station",
]
EXPECTED_SCHEMA_WITHOUT_STATION = [
    "No", "year", "month", "day", "hour", "PM2.5", "PM10", "SO2", "NO2",
    "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM",
]
CANONICAL_COLUMN_ORDER = [
    "station", "timestamp", "year", "month", "day", "hour", "No", "PM2.5",
    "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN",
    "wd", "WSPM", "source_file",
]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DATA_DIR_MODE: {RAW_DATA_DIR_MODE}")
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(RAW_DATA_DIR_NOTE)


PROJECT_ROOT: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals
RAW_DATA_DIR_MODE: canonical
RAW_DATA_DIR: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals\data\raw\beijing_multi_site_air_quality
Canonical raw path is data/raw/beijing_multi_site_air_quality/. Fallback mode is only for temporary compatibility when PRSA_Data_*.csv are placed directly under data/raw/.


## 1. Notebook scope

This notebook assembles the raw multistation dataset into one canonical structural checkpoint for downstream forecasting work.

It does **not** do EDA, imputation, split design, horizons, lags, feature engineering, baselines, leakage analysis, or modeling.

AI assistance disclosure: this notebook structure and code were AI-assisted, then reviewed against the accepted Notebook 00 design and DATA_FORGE notebook structural constraints before being written to the repository.


## 2. Dataset source and raw-input contract

Raw source: **UCI Beijing Multi-Site Air Quality Dataset**  
URL: <https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data>

Canonical raw inputs for this project are the 12 `PRSA_Data_*.csv` station files located under `data/raw/beijing_multi_site_air_quality/`.

Extra `data.csv` or `test.csv` files from the UCI package are ignored for canonical splitting and are not used by this notebook.

Raw files are treated as immutable source artifacts. This notebook only discovers, validates, assembles, and persists the canonical multistation structural checkpoint.


## 3. Raw file discovery


In [4]:
if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Raw data directory does not exist: {RAW_DATA_DIR}. "
        "Canonical path is data/raw/beijing_multi_site_air_quality/. "
        "Create the folder and place the 12 PRSA station CSV files there."
    )

if not RAW_DATA_DIR.is_dir():
    raise NotADirectoryError(f"Raw data path is not a directory: {RAW_DATA_DIR}")


In [5]:
raw_files = sorted(RAW_DATA_DIR.glob(RAW_FILE_GLOB))

if len(raw_files) != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FILE_COUNT} raw station files matching {RAW_FILE_GLOB}, "
        f"found {len(raw_files)} in {RAW_DATA_DIR}."
    )

discovered_filenames = [file_path.name for file_path in raw_files]
discovered_filenames


['PRSA_Data_Aotizhongxin_20130301-20170228.csv',
 'PRSA_Data_Changping_20130301-20170228.csv',
 'PRSA_Data_Dingling_20130301-20170228.csv',
 'PRSA_Data_Dongsi_20130301-20170228.csv',
 'PRSA_Data_Guanyuan_20130301-20170228.csv',
 'PRSA_Data_Gucheng_20130301-20170228.csv',
 'PRSA_Data_Huairou_20130301-20170228.csv',
 'PRSA_Data_Nongzhanguan_20130301-20170228.csv',
 'PRSA_Data_Shunyi_20130301-20170228.csv',
 'PRSA_Data_Tiantan_20130301-20170228.csv',
 'PRSA_Data_Wanliu_20130301-20170228.csv',
 'PRSA_Data_Wanshouxigong_20130301-20170228.csv']

## 4. Raw schema inspection


In [6]:
def load_raw_station_file(file_path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(file_path)
    except Exception as exc:
        raise RuntimeError(f"Failed to load raw file: {file_path.name}") from exc


In [7]:
raw_tables = {}

for file_path in raw_files:
    raw_tables[file_path.name] = load_raw_station_file(file_path)

len(raw_tables)


12

In [8]:
schema_by_file = {
    file_name: dataframe.columns.tolist()
    for file_name, dataframe in raw_tables.items()
}

accepted_schemas = [EXPECTED_SCHEMA_WITH_STATION, EXPECTED_SCHEMA_WITHOUT_STATION]

for file_name, schema in schema_by_file.items():
    if schema not in accepted_schemas:
        raise ValueError(
            f"Unexpected raw schema for {file_name}: {schema}"
        )

canonical_raw_schema = next(iter(schema_by_file.values()))

for file_name, schema in schema_by_file.items():
    if schema != canonical_raw_schema:
        raise ValueError(
            f"Schema mismatch for {file_name}. Expected {canonical_raw_schema}, got {schema}."
        )

files_missing_pm25 = [
    file_name
    for file_name, schema in schema_by_file.items()
    if "PM2.5" not in schema
]

if files_missing_pm25:
    raise ValueError(
        f"PM2.5 is missing from the following raw files: {files_missing_pm25}"
    )

canonical_raw_schema


['No',
 'year',
 'month',
 'day',
 'hour',
 'PM2.5',
 'PM10',
 'SO2',
 'NO2',
 'CO',
 'O3',
 'TEMP',
 'PRES',
 'DEWP',
 'RAIN',
 'wd',
 'WSPM',
 'station']

## 5. Station identity handling


In [9]:
def derive_station_from_filename(file_path: Path) -> str:
    """Extract station token from a PRSA station filename.

    Expected pattern:
      PRSA_Data_<station>_YYYYMMDD-YYYYMMDD.csv

    Example:
      PRSA_Data_Aotizhongxin_20130301-20170228.csv -> Aotizhongxin
    """

    import re

    file_name = file_path.name

    pattern = r"^PRSA_Data_(?P<station>[A-Za-z0-9]+)_(?P<start>\d{8})-(?P<end>\d{8})\.csv$"
    match = re.match(pattern, file_name)

    if match is None:
        raise ValueError(
            "Filename does not match expected PRSA pattern: "
            f"{file_name}. Expected 'PRSA_Data_<station>_YYYYMMDD-YYYYMMDD.csv'."
        )

    station_name = match.group("station").strip()

    if not station_name:
        raise ValueError(f"Filename-derived station is empty for {file_name}")

    return station_name


In [10]:
station_tables = {}
station_identity_rows = []

for file_path in raw_files:
    file_name = file_path.name
    dataframe = raw_tables[file_name].copy()
    station_from_filename = derive_station_from_filename(file_path)

    if "station" in dataframe.columns:
        observed_station_values = sorted(
            {
                str(value).strip()
                for value in dataframe["station"].dropna().unique()
            }
        )

        if len(observed_station_values) > 1:
            raise ValueError(
                f"Raw station column has multiple values in {file_name}: {observed_station_values}"
            )

        if observed_station_values and observed_station_values[0] != station_from_filename:
            raise ValueError(
                f"Filename-derived station '{station_from_filename}' does not match "
                f"in-file station '{observed_station_values[0]}' for {file_name}."
            )

    dataframe["station"] = station_from_filename
    dataframe["source_file"] = file_name
    station_tables[file_name] = dataframe

    station_identity_rows.append(
        {
            "file_name": file_name,
            "station_from_filename": station_from_filename,
            "row_count": int(len(dataframe)),
        }
    )

pd.DataFrame(station_identity_rows).sort_values("station_from_filename")


,file_name,station_from_filename,row_count
0,PRSA_Data_Aotizhongxin_20130301-20170228.csv,Aotizhongxin,35064
1,PRSA_Data_Changping_20130301-20170228.csv,Changping,35064
2,PRSA_Data_Dingling_20130301-20170228.csv,Dingling,35064
3,PRSA_Data_Dongsi_20130301-20170228.csv,Dongsi,35064
4,PRSA_Data_Guanyuan_20130301-20170228.csv,Guanyuan,35064
5,PRSA_Data_Gucheng_20130301-20170228.csv,Gucheng,35064
6,PRSA_Data_Huairou_20130301-20170228.csv,Huairou,35064
7,PRSA_Data_Nongzhanguan_20130301-20170228.csv,Nongzhanguan,35064
8,PRSA_Data_Shunyi_20130301-20170228.csv,Shunyi,35064
9,PRSA_Data_Tiantan_20130301-20170228.csv,Tiantan,35064


## 6. Timestamp reconstruction


In [11]:
def build_timestamp(dataframe: pd.DataFrame) -> pd.Series:
    required_time_columns = ["year", "month", "day", "hour"]
    missing_time_columns = [
        column_name
        for column_name in required_time_columns
        if column_name not in dataframe.columns
    ]

    if missing_time_columns:
        raise ValueError(
            f"Missing required time columns for timestamp construction: {missing_time_columns}"
        )

    timestamp = pd.to_datetime(
        dataframe[required_time_columns],
        errors="coerce",
    )

    null_timestamp_count = int(timestamp.isna().sum())
    if null_timestamp_count > 0:
        raise ValueError(
            f"Timestamp construction failed for {null_timestamp_count} rows."
        )

    return timestamp


In [12]:
timestamped_tables = {}

for file_name, dataframe in station_tables.items():
    dataframe = dataframe.copy()
    dataframe["timestamp"] = build_timestamp(dataframe)
    timestamped_tables[file_name] = dataframe

list(timestamped_tables)


['PRSA_Data_Aotizhongxin_20130301-20170228.csv',
 'PRSA_Data_Changping_20130301-20170228.csv',
 'PRSA_Data_Dingling_20130301-20170228.csv',
 'PRSA_Data_Dongsi_20130301-20170228.csv',
 'PRSA_Data_Guanyuan_20130301-20170228.csv',
 'PRSA_Data_Gucheng_20130301-20170228.csv',
 'PRSA_Data_Huairou_20130301-20170228.csv',
 'PRSA_Data_Nongzhanguan_20130301-20170228.csv',
 'PRSA_Data_Shunyi_20130301-20170228.csv',
 'PRSA_Data_Tiantan_20130301-20170228.csv',
 'PRSA_Data_Wanliu_20130301-20170228.csv',
 'PRSA_Data_Wanshouxigong_20130301-20170228.csv']

## 7. Long-format assembly


In [13]:
assembled_df = pd.concat(
    list(timestamped_tables.values()),
    axis=0,
    ignore_index=True,
)

missing_canonical_columns = [
    column_name
    for column_name in CANONICAL_COLUMN_ORDER
    if column_name not in assembled_df.columns
]
if missing_canonical_columns:
    raise ValueError(
        f"Assembled dataframe is missing canonical columns: {missing_canonical_columns}"
    )

assembled_df = assembled_df[CANONICAL_COLUMN_ORDER].copy()
assembled_df = assembled_df.sort_values(["station", "timestamp"]).reset_index(drop=True)

if assembled_df.columns.tolist() != CANONICAL_COLUMN_ORDER:
    raise ValueError("Assembled dataframe columns do not match the canonical order.")

assembled_df.head()


,station,timestamp,year,month,day,hour,No,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,source_file
0,Aotizhongxin,2013-03-01 00:00:00,2013,3,1,0,1,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,PRSA_Data_Aotizhongxin_20130301-20170228.csv
1,Aotizhongxin,2013-03-01 01:00:00,2013,3,1,1,2,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,PRSA_Data_Aotizhongxin_20130301-20170228.csv
2,Aotizhongxin,2013-03-01 02:00:00,2013,3,1,2,3,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,PRSA_Data_Aotizhongxin_20130301-20170228.csv
3,Aotizhongxin,2013-03-01 03:00:00,2013,3,1,3,4,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,PRSA_Data_Aotizhongxin_20130301-20170228.csv
4,Aotizhongxin,2013-03-01 04:00:00,2013,3,1,4,5,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,PRSA_Data_Aotizhongxin_20130301-20170228.csv


## 8. Structural validation


In [14]:
key_null_counts = {
    "station": int(assembled_df["station"].isna().sum()),
    "timestamp": int(assembled_df["timestamp"].isna().sum()),
}

duplicate_key_groups = (
    assembled_df.groupby(["station", "timestamp"], dropna=False)
    .size()
    .reset_index(name="row_count")
)
duplicate_key_groups = duplicate_key_groups[duplicate_key_groups["row_count"] > 1].copy()

duplicate_key_counts = {
    "overall": int(duplicate_key_groups["row_count"].sum()),
    "by_station": {
        station_name: int(row_count)
        for station_name, row_count in duplicate_key_groups.groupby("station")["row_count"].sum().items()
    },
}

row_counts_by_station = {
    station_name: int(row_count)
    for station_name, row_count in assembled_df.groupby("station").size().items()
}

timestamp_bounds_by_station = assembled_df.groupby("station")["timestamp"].agg(["min", "max"])
timestamp_min_by_station = {
    station_name: timestamp_value.isoformat()
    for station_name, timestamp_value in timestamp_bounds_by_station["min"].items()
}
timestamp_max_by_station = {
    station_name: timestamp_value.isoformat()
    for station_name, timestamp_value in timestamp_bounds_by_station["max"].items()
}

pm25_presence_check = {
    "present_in_all_files": len(files_missing_pm25) == 0,
    "files_missing_pm25": files_missing_pm25,
}

blocking_failures = []

if key_null_counts["station"] > 0:
    blocking_failures.append(
        f"Null station keys found: {key_null_counts['station']}"
    )

if key_null_counts["timestamp"] > 0:
    blocking_failures.append(
        f"Null timestamp keys found: {key_null_counts['timestamp']}"
    )

if duplicate_key_counts["overall"] > 0:
    blocking_failures.append(
        f"Duplicate (station, timestamp) rows found: {duplicate_key_counts['overall']}"
    )


In [15]:
validation_summary = pd.DataFrame(
    {
        "station": list(row_counts_by_station.keys()),
        "row_count": list(row_counts_by_station.values()),
        "timestamp_min": [timestamp_min_by_station[station] for station in row_counts_by_station],
        "timestamp_max": [timestamp_max_by_station[station] for station in row_counts_by_station],
        "duplicate_key_rows": [duplicate_key_counts["by_station"].get(station, 0) for station in row_counts_by_station],
    }
)

validation_summary.sort_values("station")


,station,row_count,timestamp_min,timestamp_max,duplicate_key_rows
0,Aotizhongxin,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
1,Changping,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
2,Dingling,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
3,Dongsi,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
4,Guanyuan,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
5,Gucheng,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
6,Huairou,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
7,Nongzhanguan,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
8,Shunyi,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0
9,Tiantan,35064,2013-03-01T00:00:00,2017-02-28T23:00:00,0


## 9. Persist assembled dataset and manifest


In [16]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)


In [17]:
discovered_file_inventory = []

for file_path in raw_files:
    file_name = file_path.name
    discovered_file_inventory.append(
        {
            "file_name": file_name,
            "station_from_filename": derive_station_from_filename(file_path),
            "row_count": int(len(raw_tables[file_name])),
            "column_names": schema_by_file[file_name],
        }
    )

manifest = {
    "source_dataset": {
        "name": SOURCE_DATASET_NAME,
        "url": SOURCE_DATASET_URL,
    },
    "raw_input_path": RAW_INPUT_PATH,
    "discovered_file_inventory": discovered_file_inventory,
    "schema_by_file": schema_by_file,
    "canonical_schema": {
        "columns": CANONICAL_COLUMN_ORDER,
        "key_columns": ["station", "timestamp"],
    },
    "row_counts_by_station": row_counts_by_station,
    "timestamp_min_by_station": timestamp_min_by_station,
    "timestamp_max_by_station": timestamp_max_by_station,
    "duplicate_key_counts": duplicate_key_counts,
    "key_null_counts": key_null_counts,
    "pm25_presence_check": pm25_presence_check,
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "artifact_paths": {
        "assembled_parquet": ASSEMBLED_PARQUET_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "assembly_manifest": ASSEMBLY_MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
    "promotion_status": {
        "promoted": len(blocking_failures) == 0,
        "blocking_failures": list(blocking_failures),
    },
}


In [18]:
if manifest["promotion_status"]["promoted"]:
    try:
        assembled_df.to_parquet(ASSEMBLED_PARQUET_PATH, index=False)
    except Exception as exc:
        manifest["promotion_status"]["promoted"] = False
        manifest["promotion_status"]["blocking_failures"].append(
            f"Failed to write parquet artifact: {exc}"
        )
        if ASSEMBLED_PARQUET_PATH.exists():
            ASSEMBLED_PARQUET_PATH.unlink()

with ASSEMBLY_MANIFEST_PATH.open("w", encoding="utf-8") as manifest_file:
    json.dump(manifest, manifest_file, indent=2, ensure_ascii=False)

manifest["promotion_status"]


{'promoted': True, 'blocking_failures': []}

## 10. Notebook close

Notebook 00 ends after structural assembly, validation, and conditional artifact promotion.

Notebook 01 will audit temporal gaps, missingness, station coverage, and value sanity.
